# Book Market Analytics: Books to Scrape
Data Science 500 Assignment

**Goal:** scrape book data, clean it, explore it, and predict book price with linear regression.

## 1. Data Acquisition (Web Scraping)

In [ ]:
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
BASE = "https://books.toscrape.com/"
CATALOGUE = urljoin(BASE, "catalogue/")
PAGES = 20
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

def get_soup(url):
    response = session.get(url, timeout=15)
    response.raise_for_status()
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")

**Step 1:** go through each listing page and collect title, price, rating and the link to each book.

In [ ]:
books = []
for page in range(1, PAGES + 1):
    url = urljoin(CATALOGUE, f"page-{page}.html")
    try:
        soup = get_soup(url)
        for item in soup.select("article.product_pod"):
            books.append({
                "title": item.h3.a["title"],
                "price_raw": item.select_one("p.price_color").text.strip(),
                "rating_word": item.select_one("p.star-rating")["class"][1],
                "link": urljoin(url, item.h3.a["href"])
            })
        print(f"Page {page}/{PAGES} completed. Books collected: {len(books)}")
        time.sleep(0.2)
    except requests.RequestException as error:
        print(f"Page {page} failed: {error}")

if not books:
    raise RuntimeError("No books collected. Check your internet connection.")

print("Total books collected:", len(books))

**Step 2:** visit each book page to get the genre, the stock quantity and the description.

In [ ]:
for i, book in enumerate(books, start=1):
    try:
        soup = get_soup(book["link"])
        breadcrumbs = soup.select("ul.breadcrumb li a")
        book["genre"] = breadcrumbs[2].get_text(strip=True) if len(breadcrumbs) > 2 else "Unknown"
        availability = soup.select_one("p.availability")
        book["availability"] = availability.get_text(" ", strip=True) if availability else "Unknown"
        desc = soup.select_one("#product_description")
        description = desc.find_next_sibling("p") if desc else None
        book["description"] = description.get_text(strip=True) if description else None
    except requests.RequestException as error:
        print(f"Could not retrieve {book['title']}: {error}")
        book["genre"] = "Unknown"
        book["availability"] = "Unknown"
        book["description"] = None
    if i % 10 == 0 or i == len(books):
        print(f"Processed {i}/{len(books)} books")
    time.sleep(0.1)

## 2. Data Cleaning

In [ ]:
df = pd.DataFrame(books)
print("Dataset shape:", df.shape)
print(df.isnull().sum())
print("Duplicate titles:", df.duplicated(subset="title").sum())

In [ ]:
df = df.drop_duplicates(subset="title").copy()
df["price"] = pd.to_numeric(df["price_raw"].str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating_word"].map(rating_map)
df["in_stock"] = df["availability"].str.contains("In stock", case=False, na=False)
df["quantity"] = pd.to_numeric(df["availability"].str.extract(r"\((\d+) available\)")[0], errors="coerce").fillna(0)
df["description"] = df["description"].fillna("No description")
df["desc_length"] = df["description"].str.split().str.len()
df = df[["title", "genre", "price", "rating", "rating_word", "in_stock", "quantity", "desc_length", "description"]].copy()
df.to_csv("books_dataset.csv", index=False)
df.info()
display(df.head())

**Cleaning decisions:**
- Duplicates were removed so no book is counted twice.
- Prices were converted from text (e.g. "£51.77") to numbers so they can be analysed.
- Star ratings were converted from words to numbers 1 to 5.
- Missing descriptions were filled with "No description" instead of deleting the book, to keep the dataset size.
- Missing quantities were set to 0 because they mean the book is not shown as available.

## 3. Data Visualization and Exploration

In [ ]:
print(df[["price", "rating", "quantity", "desc_length"]].describe())

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["price"].dropna(), bins=20)
plt.title("Distribution of Book Prices")
plt.xlabel("Price (GBP)")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.show()

Interpretation: *(write what you see, e.g. prices are spread evenly between about 10 and 60 pounds)*

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x="rating", y="price", data=df)
plt.title("Price by Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Price (GBP)")
plt.tight_layout()
plt.show()

Interpretation: *(do higher rated books cost more or not?)*

In [ ]:
top_genres = df["genre"].value_counts().head(10).index
avg_price = df[df["genre"].isin(top_genres)].groupby("genre")["price"].mean().sort_values()
plt.figure(figsize=(8, 5))
avg_price.plot(kind="barh")
plt.title("Average Price of the 10 Biggest Genres")
plt.xlabel("Average Price (GBP)")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

Interpretation: *(which genres are cheapest and most expensive?)*

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x="rating", data=df)
plt.title("Number of Books per Star Rating")
plt.xlabel("Star Rating")
plt.ylabel("Number of Books")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(x="quantity", y="price", data=df)
plt.title("Price vs Stock Quantity")
plt.xlabel("Stock Quantity")
plt.ylabel("Price (GBP)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x="in_stock", y="price", data=df)
plt.title("Price by Stock Availability")
plt.xlabel("Stock Availability")
plt.ylabel("Price (GBP)")
plt.xticks([0, 1], ["Out of Stock", "In Stock"])
plt.tight_layout()
plt.show()

print("Stock availability counts:")
print(df["in_stock"].value_counts())

Interpretation: *(does stock availability relate to price?)*

In [ ]:
plt.figure(figsize=(6, 4))
correlation = df[["price", "rating", "quantity", "desc_length"]].corr()
sns.heatmap(correlation, annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Between Numeric Variables")
plt.tight_layout()
plt.show()

Interpretation: *(are any variables strongly related to price? Values close to 0 mean a weak relationship)*

## 4. Predictive Modelling (Linear Regression)

**Features chosen:** star rating, stock quantity, description length and genre. These are the publicly available details of a book that could plausibly relate to price.

In [ ]:
df["genre_group"] = df["genre"].where(df["genre"].isin(top_genres), "Other")
X = pd.get_dummies(df[["rating", "quantity", "desc_length", "genre_group"]], drop_first=True, dtype=int)
y = df["price"]
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

if len(X) < 2:
    raise ValueError("Not enough valid records to train the model.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 3))

coefficients = pd.Series(model.coef_, index=X.columns).sort_values()
print("Model coefficients:")
print(coefficients)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, predictions)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color="red")
plt.xlabel("Actual Price (GBP)")
plt.ylabel("Predicted Price (GBP)")
plt.title("Actual vs Predicted Price")
plt.tight_layout()
plt.show()

print("Analysis complete. Dataset saved as books_dataset.csv")

## 5. Interpretation and Limitations
- **MAE** is the average mistake in pounds. **R2** shows how much of the price variation the model explains (0 = none, 1 = all).
- *(Write your own results here, e.g. "The model explains only X% of price variation".)*
- **Limitations:** Books to Scrape is a practice website, so prices may be close to random and not follow real market rules. Only a few features were used, no real sales data exists, and linear regression assumes straight line relationships.
- **Improvements:** add more features, try other models, use the full 1000 books.